# StreamFlix OTT Analytics — Phase 2: Pandas EDA

Loads the 6 CSVs, cleans and merges them, adds calculated columns, runs
exploratory analysis with summary reports, and generates 9 charts.

**Steps:** upload `data.zip` → run all cells → charts render inline and save
to a `charts/` folder → last cell zips and downloads them.


## 1. Upload the data
Run this cell, then choose `data.zip` when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select data.zip


In [ ]:
import zipfile, os
with zipfile.ZipFile('data.zip', 'r') as z:
    z.extractall('.')
os.makedirs('charts', exist_ok=True)
print(os.listdir('data'))


## 2. Imports & Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 20)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'

DATA_DIR = 'data'
CHART_DIR = 'charts'


## 3. Load

In [ ]:
movies = pd.read_csv(f'{DATA_DIR}/movies.csv', parse_dates=['date_added'])
users = pd.read_csv(f'{DATA_DIR}/users.csv', parse_dates=['signup_date'])
plans = pd.read_csv(f'{DATA_DIR}/subscription_plans.csv')
subs = pd.read_csv(f'{DATA_DIR}/subscriptions.csv', parse_dates=['start_date', 'end_date'])
watch = pd.read_csv(f'{DATA_DIR}/watch_history.csv', parse_dates=['watch_date'])
ratings = pd.read_csv(f'{DATA_DIR}/ratings.csv', parse_dates=['rating_date'])

for name, df in [('movies', movies), ('users', users), ('plans', plans),
                  ('subscriptions', subs), ('watch_history', watch), ('ratings', ratings)]:
    print(f'{name:15s}: {df.shape[0]:>7,} rows x {df.shape[1]} cols')


## 4. Clean

In [ ]:
movies = movies.drop_duplicates(subset=['show_id'])
movies['director'] = movies['director'].fillna('Unknown')
movies['primary_country'] = movies['primary_country'].fillna('Unknown')

watch = watch[watch['watch_duration_minutes'] > 0].copy()
watch['completion_pct'] = watch['completion_pct'].clip(0, 100)

ratings = ratings[ratings['rating'].between(1, 5)]
subs['status'] = subs['status'].str.strip().str.title()

print('Null counts (movies, top 5):')
print(movies.isnull().sum().sort_values(ascending=False).head())


## 5. Merge

In [ ]:
watch_full = (
    watch
    .merge(movies[['show_id', 'title', 'type', 'primary_genre', 'release_year']], on='show_id', how='left')
    .merge(users[['user_id', 'city', 'country', 'age', 'signup_date']], on='user_id', how='left')
)

subs_full = (
    subs
    .merge(plans, on='plan_id', how='left')
    .merge(users[['user_id', 'city', 'country']], on='user_id', how='left')
)

ratings_full = ratings.merge(movies[['show_id', 'title', 'primary_genre']], on='show_id', how='left')

print('watch_full  :', watch_full.shape)
print('subs_full   :', subs_full.shape)
print('ratings_full:', ratings_full.shape)


## 6. Calculated Columns

In [ ]:
watch_full['watch_month'] = watch_full['watch_date'].dt.to_period('M').astype(str)
watch_full['watch_year'] = watch_full['watch_date'].dt.year
watch_full['is_binge'] = watch_full['completion_pct'] >= 90

subs_full['is_active'] = subs_full['status'] == 'Active'
subs_full['tenure_days'] = (
    subs_full['end_date'].fillna(pd.Timestamp.today()) - subs_full['start_date']
).dt.days

users['age_group'] = pd.cut(
    users['age'], bins=[0, 18, 30, 45, 100],
    labels=['Teen', 'Young Adult', 'Adult', 'Senior']
)

movies['content_age_years'] = pd.Timestamp.today().year - movies['release_year']
print('Calculated columns added.')


## 7. EDA — Top 10 Most-Watched Titles

In [ ]:
top_titles = (
    watch_full.groupby('title')
    .agg(total_views=('watch_id', 'count'), total_minutes=('watch_duration_minutes', 'sum'))
    .sort_values('total_views', ascending=False)
    .head(10)
)
top_titles


## 8. EDA — Popular Genres

In [ ]:
popular_genres = (
    watch_full.groupby('primary_genre')
    .agg(view_count=('watch_id', 'count'), total_minutes=('watch_duration_minutes', 'sum'))
    .sort_values('total_minutes', ascending=False)
)
popular_genres.head(10)


## 9. EDA — Active Cities

In [ ]:
active_cities = (
    watch_full.groupby(['city', 'country'])
    .agg(watch_events=('watch_id', 'count'), active_users=('user_id', 'nunique'))
    .sort_values('watch_events', ascending=False)
    .head(10)
)
active_cities


## 10. EDA — Revenue by Plan

In [ ]:
revenue_by_plan = (
    subs_full.groupby('plan_name')
    .agg(subscription_count=('subscription_id', 'count'), total_revenue=('monthly_revenue', 'sum'))
    .sort_values('total_revenue', ascending=False)
)
revenue_by_plan


## 11. EDA — Device Usage

In [ ]:
device_usage = (
    watch_full.groupby('device')
    .agg(sessions=('watch_id', 'count'), total_minutes=('watch_duration_minutes', 'sum'),
         avg_minutes=('watch_duration_minutes', 'mean'))
    .sort_values('sessions', ascending=False)
)
device_usage.round(1)


## 12. EDA — Top-Rated Movies (min 5 ratings)

In [ ]:
rating_stats = (
    ratings_full.groupby(['show_id', 'title'])
    .agg(num_ratings=('rating', 'count'), avg_rating=('rating', 'mean'))
)
top_rated = rating_stats[rating_stats['num_ratings'] >= 5].sort_values(
    ['avg_rating', 'num_ratings'], ascending=False
).head(10)
top_rated.round(2)


## 13. EDA — Monthly Viewing Trend

In [ ]:
monthly_trend = (
    watch_full.groupby('watch_month')
    .agg(total_views=('watch_id', 'count'), total_minutes=('watch_duration_minutes', 'sum'),
         unique_viewers=('user_id', 'nunique'))
    .sort_index()
)
monthly_trend.tail(12)


## 14. EDA — Churn-Risk Users

In [ ]:
last_watch = watch_full.groupby('user_id')['watch_date'].max().rename('last_watch_date')
active_users = subs_full[subs_full['is_active']]['user_id'].unique()
churn = pd.DataFrame({'user_id': active_users}).merge(last_watch, on='user_id', how='left')
churn['days_since_last_watch'] = (pd.Timestamp.today() - churn['last_watch_date']).dt.days
churn['churn_risk'] = pd.cut(
    churn['days_since_last_watch'], bins=[-1, 30, 60, 100000],
    labels=['Low', 'Medium', 'High']
)
churn['churn_risk'].value_counts()


## 15. Charts
Each cell renders inline and saves a PNG to `charts/`.

### Top 10 Most-Watched Titles

In [ ]:
fig, ax = plt.subplots()
top_titles['total_views'].sort_values().plot(kind='barh', ax=ax, color='#E50914')
ax.set_title('Top 10 Most-Watched Titles')
ax.set_xlabel('Total Views')
fig.tight_layout()
fig.savefig(f'{CHART_DIR}/01_top_watched_titles.png', dpi=120)
plt.show()


### Top 10 Genres by Watch Minutes

In [ ]:
fig, ax = plt.subplots()
popular_genres.head(10)['total_minutes'].sort_values().plot(kind='barh', ax=ax, color='#221F1F')
ax.set_title('Top 10 Genres by Total Watch Minutes')
ax.set_xlabel('Total Minutes')
fig.tight_layout()
fig.savefig(f'{CHART_DIR}/02_popular_genres.png', dpi=120)
plt.show()


### Revenue Share by Plan

In [ ]:
fig, ax = plt.subplots()
ax.pie(revenue_by_plan['total_revenue'], labels=revenue_by_plan.index, autopct='%1.1f%%',
       colors=['#E50914', '#B81D24', '#221F1F'])
ax.set_title('Revenue Share by Subscription Plan')
fig.tight_layout()
fig.savefig(f'{CHART_DIR}/03_revenue_by_plan.png', dpi=120)
plt.show()


### Sessions by Device

In [ ]:
fig, ax = plt.subplots()
device_usage['sessions'].plot(kind='bar', ax=ax, color='#E50914')
ax.set_title('Watch Sessions by Device')
ax.set_ylabel('Sessions')
plt.xticks(rotation=0)
fig.tight_layout()
fig.savefig(f'{CHART_DIR}/04_device_usage.png', dpi=120)
plt.show()


### Monthly Viewing Trend

In [ ]:
fig, ax = plt.subplots()
monthly_trend['total_views'].plot(kind='line', ax=ax, marker='o', color='#E50914')
ax.set_title('Monthly Viewing Trend')
ax.set_ylabel('Total Views')
ax.set_xlabel('Month')
plt.xticks(rotation=90, fontsize=7)
fig.tight_layout()
fig.savefig(f'{CHART_DIR}/05_monthly_trend.png', dpi=120)
plt.show()


### Top 10 Active Cities

In [ ]:
fig, ax = plt.subplots()
active_cities['watch_events'].sort_values().plot(kind='barh', ax=ax, color='#221F1F')
ax.set_title('Top 10 Active Cities by Watch Events')
ax.set_xlabel('Watch Events')
fig.tight_layout()
fig.savefig(f'{CHART_DIR}/06_active_cities.png', dpi=120)
plt.show()


### Churn Risk Distribution

In [ ]:
fig, ax = plt.subplots()
churn['churn_risk'].value_counts().reindex(['Low', 'Medium', 'High']).plot(
    kind='bar', ax=ax, color=['#2ecc71', '#f39c12', '#e74c3c']
)
ax.set_title('Churn Risk Distribution (Active Subscribers)')
ax.set_ylabel('User Count')
plt.xticks(rotation=0)
fig.tight_layout()
fig.savefig(f'{CHART_DIR}/07_churn_risk.png', dpi=120)
plt.show()


### User Distribution by Age Group

In [ ]:
fig, ax = plt.subplots()
users['age_group'].value_counts().reindex(['Teen', 'Young Adult', 'Adult', 'Senior']).plot(
    kind='bar', ax=ax, color='#B81D24'
)
ax.set_title('User Distribution by Age Group')
ax.set_ylabel('User Count')
plt.xticks(rotation=0)
fig.tight_layout()
fig.savefig(f'{CHART_DIR}/08_age_groups.png', dpi=120)
plt.show()


### Movies vs TV Shows Watch Share

In [ ]:
fig, ax = plt.subplots()
watch_full['type'].value_counts().plot(kind='pie', ax=ax, autopct='%1.1f%%',
                                        colors=['#E50914', '#221F1F'])
ax.set_title('Watch Events: Movies vs TV Shows')
ax.set_ylabel('')
fig.tight_layout()
fig.savefig(f'{CHART_DIR}/09_movie_vs_tv_watch_share.png', dpi=120)
plt.show()


## 16. Download all charts

In [ ]:
import shutil
shutil.make_archive('charts', 'zip', 'charts')
from google.colab import files
files.download('charts.zip')
